# Tune `linear_lr`

Linear (RF top-k + T² + hub×hub interactions) + elastic-net logistic (saga). Repeated stratified CV on the train split;
writes [`data/processed/tuned/linear_lr.json`](../data/processed/tuned/linear_lr.json).

Tune all models: [`tune_all.ipynb`](tune_all.ipynb).


**Classifier:** `elastic_net_lr` (saga + `penalty="elasticnet"`) in `src/secom/pipelines.py`.

**Stage 1 (hyperparameters):** RF top-k (`top_k`), hub count (`n_hubs`), classifier `C`, `l1_ratio`. Select by **max mean PR AUC**.

**Stage 2 (threshold):** sweep `THRESHOLD_GRID` on the same CV folds; select threshold that **minimizes mean BER**.

**Shared preprocess:** median impute → Spearman cluster → RF top-k → Hotelling T² → hub pair interactions (auxiliary features pass through).


In [1]:
import importlib

import pandas as pd


import secom.tuning.registry as tuning_registry

importlib.reload(tuning_registry)
from secom.pipelines import TARGET_COL, feature_columns, load_mart, split_train_test
from secom.tuning.registry import (
    MODEL_SPECS,
    fit_with_progress,
    run_grid_search,
    save_tuned_params,
    summarize_cv_search,
    tune_classifier_threshold_profiles,
    tuned_params_path,
)

MODEL_ID = "linear_lr"
spec = MODEL_SPECS[MODEL_ID]


In [2]:
df = load_mart()
feature_cols = feature_columns(df)
train_df, test_df = split_train_test(df)
X_train = train_df[feature_cols]
y_train = train_df[TARGET_COL].astype(int)
print(len(X_train), "train rows", len(test_df), "test rows (holdout, not used here)")


1253 train rows 314 test rows (holdout, not used here)


In [3]:
param_grid = spec.make_param_grid()
pd.DataFrame([{k: v} for k, v in param_grid.items()])


,preprocess__sensor_branch__select_t2_hubs__top_k,preprocess__sensor_branch__select_t2_hubs__n_hubs,preprocess__sensor_branch__cluster__smart_corr__threshold,classifier__estimator__C,classifier__estimator__l1_ratio
0,[35],NaN,NaN,NaN,NaN
1,NaN,[5],NaN,NaN,NaN
2,NaN,NaN,[0.85],NaN,NaN
3,NaN,NaN,NaN,"[0.005, 0.0075]",NaN
4,NaN,NaN,NaN,NaN,"[0.3, 0.4, 0.5]"


In [4]:
search, n_candidates, n_splits, total_fits = run_grid_search(spec, X_train, y_train)
print(f"{MODEL_ID}: {n_candidates} candidates x {n_splits} folds = {total_fits} fits")
search = fit_with_progress(search, X_train, y_train)


linear_lr: 6 candidates x 10 folds = 60 fits


GridSearchCV 60 fits:   0%|          | 0/60 [00:00<?, ?it/s]

  0%|          | 0/60 [00:00<?, ?it/s]

Fitting 10 folds for each of 6 candidates, totalling 60 fits


In [5]:
cv_summary, fold_results, aggregated = summarize_cv_search(search, spec)
print("Stage 1 best (mean PR AUC):")
display(aggregated.head(10))


Stage 1 best (mean PR AUC):


,top_k,n_hubs,corr_threshold,c,l1_ratio,mean_ber_percent,std_ber_percent,mean_balanced_accuracy,mean_true_positive_percent,std_true_positive_percent,mean_true_negative_percent,std_true_negative_percent,mean_roc_auc,std_roc_auc,mean_pr_auc,std_pr_auc
3,35,5,0.85,0.0075,0.3,50.042735,0.128205,0.499573,0.000000,0.000000,99.914530,0.256410,0.728132,0.034705,0.174265,0.039172
4,35,5,0.85,0.0075,0.4,50.085470,0.170940,0.499145,0.000000,0.000000,99.829060,0.341880,0.726320,0.037600,0.170525,0.031048
5,35,5,0.85,0.0075,0.5,50.170940,0.391673,0.498291,0.000000,0.000000,99.658120,0.783346,0.710266,0.039829,0.165371,0.035315
0,35,5,0.85,0.0050,0.3,50.170940,0.391673,0.498291,0.000000,0.000000,99.658120,0.783346,0.711373,0.039904,0.165202,0.032505
1,35,5,0.85,0.0050,0.4,49.879808,1.007851,0.501202,0.625000,1.875000,99.615385,0.775143,0.706390,0.046185,0.159133,0.025399
2,35,5,0.85,0.0050,0.5,49.585690,1.148538,0.504143,1.213235,2.427863,99.615385,0.700907,0.695177,0.035257,0.148765,0.022916


In [6]:
threshold_result = tune_classifier_threshold_profiles(spec, X_train, y_train, cv_summary)
print("Stage 2 — F-beta thresholds (F1 conservative / F2 neutral / F3 aggressive):")
for pid, prof in threshold_result["profiles"].items():
    print(
        f"  {pid}: threshold={prof['best_threshold']:.4f}, "
        f"mean_fbeta={prof['mean_fbeta']:.4f}, "
        f"mean_ber={prof['mean_ber_percent']:.2f}%"
    )
print(
    f"Deploy (F2): threshold={threshold_result['best_threshold']:.4f}, "
    f"mean_fbeta={threshold_result['mean_fbeta']:.4f}"
)
display(threshold_result["objective_curves"].head(10))
if "per_threshold_mean_ber" in threshold_result:
    display(threshold_result["per_threshold_mean_ber"].head(10))


Threshold CV folds:   0%|          | 0/10 [00:00<?, ?it/s]

Stage 2 — F-beta thresholds (F1 conservative / F2 neutral / F3 aggressive):
  f1: threshold=0.1309, mean_fbeta=0.2548, mean_ber=34.33%
  f2: threshold=0.0859, mean_fbeta=0.3805, mean_ber=31.38%
  f3: threshold=0.0420, mean_fbeta=0.4864, mean_ber=33.63%
Deploy (F2): threshold=0.0859, mean_fbeta=0.3805


,threshold,mean_ber_percent,mean_fbeta_f1,mean_fbeta_f2,mean_fbeta_f3
0,0.001000,0.129407,0.129407,0.270897,0.426266
1,0.001999,0.129407,0.129407,0.270897,0.426266
2,0.002998,0.130469,0.130469,0.272434,0.427548
3,0.003997,0.130469,0.130469,0.272434,0.427548
4,0.004996,0.130517,0.130517,0.272520,0.427656
5,0.005995,0.131291,0.131291,0.273875,0.429334
6,0.006994,0.131444,0.131444,0.274145,0.429669
7,0.007993,0.132660,0.132660,0.276275,0.432306
8,0.008992,0.133169,0.133169,0.277165,0.433405
9,0.009991,0.132310,0.132310,0.274802,0.428771


,threshold,mean_ber_percent
425,0.425575,0.0
426,0.426574,0.0
427,0.427573,0.0
428,0.428572,0.0
429,0.429571,0.0
430,0.430570,0.0
431,0.431569,0.0
432,0.432568,0.0
433,0.433567,0.0
434,0.434566,0.0


In [7]:
payload = save_tuned_params(
    spec,
    cv_summary,
    fold_results,
    aggregated,
    threshold_result=threshold_result,
)
out_path = tuned_params_path(MODEL_ID)
print(f"Wrote {out_path}")
payload["grid_search_best_params"]


Wrote /home/troy/SECOM/data/processed/tuned/linear_lr.json


{'preprocess__sensor_branch__select_t2_hubs__top_k': 35,
 'preprocess__sensor_branch__select_t2_hubs__n_hubs': 5,
 'preprocess__sensor_branch__cluster__smart_corr__threshold': 0.85,
 'classifier__estimator__C': 0.0075,
 'classifier__estimator__l1_ratio': 0.3}